# Rare Cell Type Analysis — Part 02: Integrate and Export (GPU)

This notebook integrates the annotated region subset from Part 01 across datasets,
refines the labels, and exports the final dataset.

The workflow:

1. Integrates datasets with scVI.
2. Refines labels with scANVI, using the Part 01 `seed_label` column as seed labels.
3. Compares scVI, scANVI (and a Harmony baseline) for how well they separate the
   target population, and measures agreement with the Part 01 labels.
4. Exports the annotated dataset, marker concordance tables, and figures.

> **Run in:** GPU app (T4). Train with `accelerator="gpu"`.
> **Input:** annotated AnnData from Part 01, loaded from the workspace bucket.
> **Output:** final annotated AnnData and summary tables.

## Imports, config, and paths

Keep `ANALYSIS_NAME`, `BATCH_KEY`, and `TARGET_LABEL` identical to Part 01.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch
from google.cloud import storage
import scvi

ANALYSIS_NAME = "sn_dopaminergic"
BATCH_KEY     = "ASAP_dataset_id"
TARGET_LABEL  = "Dopaminergic"
RANDOM_SEED   = 42

OUTPUT_DIR = Path.home() / "workspace" / "ws_files" / ANALYSIS_NAME / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GPU available:", torch.cuda.is_available(), "|", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU: check the app type and the torch CUDA build (see README)"


## Load the Part 01 object from the workspace bucket

In [ ]:
ANNOTATED = OUTPUT_DIR / f"{ANALYSIS_NAME}__01_annotated.h5ad"
adata_full = ad.read_h5ad(ANNOTATED)

print(f"{adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes")

# scVI trains on the HVG subset + raw counts, both prepared in the prep notebook.
assert "counts" in adata_full.layers, "expected a 'counts' layer with raw counts"
assert "highly_variable" in adata_full.var, "expected HVGs flagged in the prep notebook"
hvg = adata_full[:, adata_full.var["highly_variable"]].copy()
print("HVG genes:", hvg.n_vars)

## scVI integration

In [ ]:
adata_full_IN   = OUTPUT_DIR / "adata_full_for_scvi.h5ad"   # written by the prep notebook
LATENT_OUT   = OUTPUT_DIR / "X_scVI.npy"              # this notebook writes this
OBSNAMES_OUT = OUTPUT_DIR / "X_scVI_obs_names.csv"    # row order guard
MODEL_OUT    = OUTPUT_DIR / "scvi_model"              # trained model (optional reuse)

In [ ]:
scvi.settings.seed = RANDOM_SEED 

scvi.model.SCVI.setup_anndata(
    hvg,
    layer="counts",
    batch_key="dataset",
    continuous_covariate_keys=["pct_counts_mt"],
)

scvi.settings.seed = RANDOM_SEED 
model = scvi.model.SCVI(hvg, n_latent=30)

model.train(
    accelerator="gpu",
    devices=1,
    early_stopping=True,   # stop when validation loss plateaus
    batch_size=1024,       # lower to 512 if you hit CUDA out-of-memory
)

### save the latent representation

In [ ]:
latent = model.get_latent_representation()          # (n_cells, 30)
np.save(LATENT_OUT, latent)

# Save cell order alongside it so the downstream notebook can align exactly.
pd.Series(hvg.obs_names, name="obs_name").to_csv(OBSNAMES_OUT, index=False)

# Optionally persist the trained model for reuse / label transfer later.
model.save(str(MODEL_OUT), overwrite=True)

print("latent:", latent.shape, "->", LATENT_OUT)
print("model  ->", MODEL_OUT)

## scANVI label refinement

**Optional extension, not required.** If you want to refine labels and propagate them
to `Unknown` cells, scANVI extends scVI using the MapMyCells labels as supervision. It
needs a trained scVI *model* and a GPU/MPS kernel, so it's fenced off from the main
path. Skip it entirely and the pipeline is complete without it

In [ ]:
latent = np.load(OUTPUT_DIR / "X_scVI.npy")
latent_names = pd.read_csv(OUTPUT_DIR / "X_scVI_obs_names.csv")["obs_name"].astype(str).values

scvi_order = pd.Index(latent_names).get_indexer(adata_full.obs_names)
assert (scvi_order >= 0).all(), "some cells in sn_full are missing from the saved latent"
adata_full.obsm["X_scVI"] = latent[scvi_order]
print("X_scVI restored:", adata_full.obsm["X_scVI"].shape)

In [ ]:
RUN_SCANVI = True   # True on a GPU/MPS kernel

if RUN_SCANVI:
    import scvi
    hvg = adata_full[:, adata_full.var["highly_variable"]].copy()
    scvi.model.SCVI.setup_anndata(hvg, layer="counts", batch_key="dataset")
    scvi_model = scvi.model.SCVI(hvg, n_latent=30)
    scvi_model.train(early_stopping=True, batch_size=1024, accelerator="gpu" )     

    hvg.obs["seed_labels"] = adata_full.obs.loc[hvg.obs_names, "seed_label"].astype(str).values
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_model, adata=hvg, labels_key="seed_labels", unlabeled_category="Unknown")
    scanvi_model.train(max_epochs=20, batch_size=1024, accelerator="gpu")       

    adata_full.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
    adata_full.obs["C_scANVI"] = scanvi_model.predict()
    # basal-ganglia taxonomy calls dopaminergic cells "M Dopa"
    print("scANVI DA calls:", (adata_full.obs["C_scANVI"] == TARGET_LABEL).sum())
else:
    print("scANVI skipped (RUN_SCANVI=False)")

## Harmony baseline and scib-metrics comparison


## Harmony

Harmony corrects the PCA embedding in minutes on CPU. It's here as a *comparison* for
scib, not the primary integration.

In [ ]:
import harmonypy

sc.pp.pca(adata_full, n_comps=30)

ho = harmonypy.run_harmony(
    adata_full.obsm["X_pca"],          # (n_cells, 30)
    adata_full.obs,
    vars_use=[BATCH_KEY],
)

# harmonypy's Z_corr should be (dims, n_cells); coerce to a real 2-D array and orient
Z = np.asarray(ho.Z_corr)
print("raw Z_corr shape:", Z.shape)

# want (n_cells, 30): transpose if it came back (30, n_cells)
if Z.shape[0] == adata_full.n_obs:
    harmony_emb = Z
elif Z.shape[1] == adata_full.n_obs:
    harmony_emb = Z.T
else:
    raise ValueError(f"unexpected Z_corr shape {Z.shape} for {adata_full.n_obs} cells")

adata_full.obsm["X_pca_harmony"] = harmony_emb
print("harmony:", adata_full.obsm["X_pca_harmony"].shape)   # (692895, 30)

## Validate integration with scib-metrics

**Did integration actually work?** scib-metrics scores each embedding on the tradeoff
between *batch mixing* (do datasets blend?) and *bio-conservation* (are cell types
preserved?). We compare the unintegrated PCA baseline, Harmony, and scVI, using the
MapMyCells class as the biological label. **This is essential validation, not optional** 

In [ ]:

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

label_key = "mmc_class_name"
lbl = adata_full.obs[label_key].astype(str)
keep = (lbl.notna() & (lbl != "nan")).values

is_da = (adata_full.obs["cell_type"] == TARGET_LABEL).values & keep
rest_pool = np.where((~is_da) & keep)[0]
rest = np.random.choice(rest_pool, size=min(45_000, rest_pool.size), replace=False)
idx = np.concatenate([np.where(is_da)[0], rest])
sub = adata_full[idx].copy()
print(f"subsample: {sub.n_obs:,} cells ({int(is_da.sum()):,} DA)")

embeddings = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI"] if k in sub.obsm]
print("benchmarking:", embeddings)

bm = Benchmarker(
    sub, batch_key=BATCH_KEY, label_key=label_key,
    embedding_obsm_keys=embeddings,
    bio_conservation_metrics=BioConservation(isolated_labels=False),
    batch_correction_metrics=BatchCorrection(kbet_per_label=False),
    n_jobs=-1,
)
bm.benchmark()
bm.plot_results_table()

In [ ]:
adata_full.obs["cell_type_refined"] = hvg.obs["C_scANVI"].reindex(adata_full.obs_names).values

## Label agreement and export

In [ ]:
# MapMyCells vs scANVI agreement for the target population
ct = pd.crosstab(adata_full.obs["cell_type"], adata_full.obs["cell_type_refined"])
print(ct.loc[[TARGET_LABEL]] if TARGET_LABEL in ct.index else ct)

In [ ]:
# -- per-sample summary: the table researchers need before any comparison --------
per_sample = (adata_full.obs
    .groupby([BATCH_KEY, "sample_id", "ASAP_subject_id", "condition_id"], observed=True)
    .agg(n_cells=("is_target", "size"), n_target=("is_target", "sum"))
    .assign(frac_target=lambda d: d.n_target / d.n_cells)
    .reset_index())
PER_SAMPLE_CSV = OUTPUT_DIR / f"{ANALYSIS_NAME}__02_per_sample_summary.csv"
per_sample.to_csv(PER_SAMPLE_CSV, index=False)

# -- benchmark results (scib + target neighborhood) ------------------------------
BENCH_CSV = OUTPUT_DIR / f"{ANALYSIS_NAME}__02_integration_benchmark.csv"
bench = bm.get_results(min_max_scale=False)
bench.to_csv(BENCH_CSV)

# -- provenance ------------------------------------------------------------------
adata_full.uns["scvi_provenance"] = {
    "seed": RANDOM_SEED, "batch_key": BATCH_KEY, "n_latent": 30,
    "scvi_tools": scvi.__version__, "torch": torch.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
}
adata_full.uns["analysis"] = {"name": ANALYSIS_NAME, "target_label": TARGET_LABEL,
                              "primary_embedding": "X_scVI"}

In [ ]:

# -- save models, final object, tables; upload -----------------------------------
for m, tag in [(scvi_model, "scvi"), (scanvi_model, "scanvi")]:
    d = OUTPUT_DIR / f"{ANALYSIS_NAME}__{tag}_model"
    m.save(d, overwrite=True, save_anndata=False)


FINAL = OUTPUT_DIR / f"{ANALYSIS_NAME}__02_final.h5ad"
adata_full.write_h5ad(FINAL)
print(f"final: {adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes")

## Next steps

The exported object is the hand-off point. Before using it for disease comparisons,
read **Known limitations** in the README: at the time of running DA neurons come almost entirely from
DS_PMDBS_0006, and DA fraction varies widely between samples, so DA proportion is
not a valid readout of DA loss.

Recommended starting points: pseudobulk DE within DA neurons (per sample, PD vs.
control), DA subtype composition among DA cells, and cell-state analyses in more
balanced populations. Use `layers["counts"]` and `obsm["X_scVI"]`.